# E-Commerce Agentic Bots
1. 📦 Global Products Agent
- `Tools:` Web Scrapes from popular webpages such as Amazon, Spotify in `json-like format`
- `Description:` Writes a description about each product as well as their price, adhers to a specific pydantic format
- `PDF:` Saves Description about each item in a single PDF report, that can be vectorised and stored in a `vector database`

2. 🐼Chinese Latest Trend Agent
- `Tools:` API tools that fetches in `json-like format` or use `Tarvily` for web search smartly, an optimisted web-search library for LLM
- `Description:` Writes a summary about the latest news in china regarding products, general trends
- `PDF:` Saves the summary in a single PDF report, that can be vectorised and stored in a `vector database`

3. 🏛️📜Chinese Legal Platform, Commerical, Legal laws Agent
- `Tools:` uses `Tarvily` for web search smartly
- `Description:` Saves all these legal laws in a single document without reading them
- `PDF:` Saves all legal information in a single PDF report, that can be vectorised and stored in a `vector database`

4. ⛩️Chinese Cultural Nuance Creator Agent
- `Tools:` Access all PDF reports using `RAG` to generate a general report for chinese audience
- `PDF:` Saves a PDF report, that can be vectorised and stored in a `vector database`

5. 🎥Platform Specific Agent
- `Tools:` Access PDF generated by `Chinese Cultural Nuance Creator Agent` using `RAG` technique
- `Generation:` Generates a script for each specific platform separately

6. 🏮Translator Agent
- `Tools:` Access PDF generated by  Platform Specific Agent, translates it into several chinese dialects including Mandarin

# Product Agent
1. Webpages to scrape `http://books.toscrape.com/`
2. Made up of 3 Nodes (Web Scrapping)
    - Node A (Fetch and Minimise) given a webpage to be scrapped, it will scrape it and using soap to remove header, ads,  style, script, etc.. then convert it to `MarkDown` format instead of `html` , `MarkDown` is recommended format of data to be fed to agent
    - Node B (Extract) feeding output from Node A to Node B to and ask it to adhere to defined Product Schema
    - Node C (The Router/Validator) loops back to Node A if it did not successfully fetch data
3. Records How many calls, time taken to finish the task, also things that slowed down agent while working
4. API to be called `https://api.escuelajs.co/api/v1/products`
5. Made up of 2 Nodes (API)
    - Node A (Fetch and Match) given API json data, some of titles might not match with data, agent job is to make it match with current schema
    - Node B (The Router/Validator) loops back to Node A if it did not successfully fetch data
6. Records How many calls, time taken to finish the task, also things that slowed down agent while working

    

In [2]:
WEB_PAGES_SCRAPE=["https://api.escuelajs.co/api/v1/products","http://books.toscrape.com/"]

In [20]:
from pydantic import BaseModel, Field
from typing import List
from typing_extensions import Annotated, TypedDict
from langgraph.graph.message import add_messages

class Product(BaseModel):
    name: str
    price: float
    description: str
    category: str | None = Field(description="name of the category the product belongs to")
    rating: float | None = Field(description="extract the rating of the product if it only exists, do not make it up")

class ProductCatalog(BaseModel):
    products : List[Product] = Field(description="A list of e-commerce products")

class URLSchema(BaseModel):
    web_scrape_url: str | None = Field(description="only url thats doesn't contain word api in them, assume they are to web scrape url, write that url here only",
                                examples=["http://example.com"])
    api_url: str | None = Field(description="Urls that contain api term in them assume write that url here only",
                         examples=["https://api.postman.com"])


def merge_products(current: List[Product], new: List[Product]) -> List[Product]:
    if not current:
        return new
    merged_dict = current | new
    return merged_dict
    


class State(TypedDict):
    messages: Annotated[list[str], add_messages]
    products_list: Annotated[list[Product], merge_products]

# Scrapping + API Sub-Graph

In [21]:
from langchain.chat_models import init_chat_model

extract_product_llm = init_chat_model(model="gemini-3.1-flash-lite", model_provider="google_genai")

In [22]:
from requests import request
from bs4 import BeautifulSoup
from html_to_markdown import convert


def fetch_raw_html(state: State):
    """Given URL it web scrapes to get html, this html is cleaned and unwanted tags are removed"""
    print("state is",state["messages"])
    last_message = state["messages"][-1].content
    extract_url_llm = extract_product_llm.with_structured_output(URLSchema)
    results = extract_url_llm.invoke([{
        "role":"system","content":"Get string of URL only",
        "role":"user","content":last_message
    }])
    url = results.web_scrape_url
    print("url is",url)
    # url="http://books.toscrape.com/"
    exclude_tags=["script","head","title","style","svg","!doctype","meta"]
    try:
        response = request(method="GET",url=url)
        if response.status_code == 200:
            soup =  BeautifulSoup(response.text, "html.parser")

            for tag in exclude_tags:
                for match in soup.find_all(tag):
                    match.extract()

            return {"messages":str(soup)}

        else:
            return f"Error Failed to fetch raw HTML from fetch_raw_html() Reason : Response_Code is  {response.status_code}"
    except Exception as e:
        return f"Error Failed to fetch raw HTML from fetch_raw_html() Reason : {e}"


def clean_to_markdown(state: State):
    """the cleaned html from fetch_raw_html() is then converted to markdown format"""
    html_content = state["messages"][-1].content
    markdown_object = convert(str(html_content))
    markdown = markdown_object.content
    return {"message":markdown}

def extract_structured_data(state: State):
    """Given a markdown as input, it is taken to extract products information according to pydantic Schema format"""
    markdown = state["messages"][-1].content
    structured_output_llm = extract_product_llm.with_structured_output(ProductCatalog)
    msg = [
        {"role":"system","content":"extract the text from markdown and give output in pydantic specified format"},
        {"role":"user","content":markdown}
    ]
    product_cat_object = structured_output_llm.invoke(msg)
    print(product_cat_object)
    print(product_cat_object.products)
    return {"products_list",product_cat_object.products}
    

def fetch_api_data():
    pass

def extract_api_structured_data():
    pass

def validate_extraction():
    pass

In [23]:
from langgraph.graph import StateGraph, START, END


graph = StateGraph(State)
graph.add_node("fetch_raw_html",fetch_raw_html)
graph.add_node("clean_to_markdown",clean_to_markdown)
graph.add_node("extract_structured_data",extract_structured_data)

graph.add_edge(START,"fetch_raw_html")
graph.add_edge("fetch_raw_html","clean_to_markdown")
graph.add_edge("clean_to_markdown","extract_structured_data")
graph.add_edge("extract_structured_data",END)
compiled_graph = graph.compile()



In [24]:
inputs = {
    "messages": [
        {
            "role": "user", 
            "content": "Use this url to extract at only 10 books http://books.toscrape.com/"
        }
    ]
}

compiled_graph.invoke(
inputs
)

state is [HumanMessage(content='Use this url to extract at only 10 books http://books.toscrape.com/', additional_kwargs={}, response_metadata={}, id='3ad4186a-a750-44d9-83e0-fb8423fffe21')]
url is http://books.toscrape.com/
products=[Product(name='A Light in the Attic', price=51.77, description='A Light in the Attic', category=None, rating=None), Product(name='Tipping the Velvet', price=53.74, description='Tipping the Velvet', category=None, rating=None), Product(name='Soumission', price=50.1, description='Soumission', category=None, rating=None), Product(name='Sharp Objects', price=47.82, description='Sharp Objects', category=None, rating=None), Product(name='Sapiens: A Brief History of Humankind', price=54.23, description='Sapiens: A Brief History of Humankind', category=None, rating=None), Product(name='The Requiem Red', price=22.65, description='The Requiem Red', category=None, rating=None), Product(name='The Dirty Little Secrets of Getting Your Dream Job', price=33.34, description

TypeError: unhashable type: 'list'